# Parts 1–2 Demo: TESS Preprocessing + Periodic Dip Detection

This notebook demonstrates the first two parts of the exoplanet light-curve project.

**Part 1:** Load or synthesize a TESS-like light curve, choose SAP/PDCSAP safely, quality-mask, normalize, detrend, compute QC metrics, and plot preprocessing diagnostics.

**Part 2:** Run periodic dip detection, estimate period/duration/depth/SNR, and generate candidate diagnostics.

The synthetic example is only for testing the pipeline. Real inference should use local TESS FITS files or MAST downloads. The loader never silently replaces failed real data with synthetic data.

In [ ]:
from pathlib import Path
import sys

# If running inside this repository without installing it:
sys.path.append(str(Path.cwd() / "src"))

from exoplanet_pipeline.config import PipelineConfig
from exoplanet_pipeline.synthetic import make_synthetic_transit_lc
from exoplanet_pipeline.preprocess import preprocess_raw_lightcurve, save_clean_lightcurve
from exoplanet_pipeline.detect import detect_candidates, candidates_to_dataframe
from exoplanet_pipeline.diagnostics import plot_preprocessing, plot_detection

## Configuration

For first-pass work we use BLS because it is fast and easier to debug. After the foundation works, switch `detection_method="tls"` if `transitleastsquares` is installed.

In [ ]:
cfg = PipelineConfig(
    detection_method="bls",
    n_periods=3000,
    detrend_method="rolling_median",
    detrend_window_days=1.0,
    make_plots=True,
)
cfg

## Part 1 demo using synthetic TESS-like light curve

The true injected signal is:

- period = 3 days
- depth = 1000 ppm
- duration = 2 hours

This lets us verify that the preprocessing does not erase the signal.

In [ ]:
raw = make_synthetic_transit_lc(
    period_days=3.0,
    depth_ppm=1000,
    duration_hours=2.0,
    noise_ppm=300,
    random_seed=42,
)

clean = preprocess_raw_lightcurve(raw, cfg)
print(clean.status)
print(clean.warnings)
clean.qc

In [ ]:
fig = plot_preprocessing(clean)

## Save Part 1 outputs

The saved files contain cleaned arrays and metadata/QC metrics. For real batch processing, later stages should reuse these processed files instead of reopening raw FITS every time.

In [ ]:
data_path, meta_path = save_clean_lightcurve(clean, "data/processed")
print(data_path)
print(meta_path)

## Part 2: periodic dip detection

Detection returns candidate signals. It does **not** claim planet/EB/blend classification yet.

In [ ]:
result = detect_candidates(clean, cfg, use_variants=False)
print(result.status)
print(result.best_candidate)

In [ ]:
candidates_df = candidates_to_dataframe(result)
candidates_df

In [ ]:
fig = plot_detection(clean, result)

## Running on a real local TESS FITS file

Uncomment and change the path below when you have a downloaded TESS light-curve FITS file.

The real-data path never falls back to synthetic data. If loading fails, the status will explicitly say so.

In [ ]:
# from exoplanet_pipeline.preprocess import preprocess_fits_file
# real_clean = preprocess_fits_file("path/to/tess_lightcurve.fits", cfg)
# print(real_clean.status, real_clean.warnings)
# plot_preprocessing(real_clean)
# real_result = detect_candidates(real_clean, cfg)
# candidates_to_dataframe(real_result)

## What comes after Parts 1–2

The candidate table should feed later stages:

1. parameter refinement
2. odd/even depth test
3. secondary eclipse search
4. centroid/blend features
5. stellar variability/systematic features
6. rule-based baseline classifier
7. AI classifier trained on curated labels
8. uncertainty estimation and validation